# Tập 4.6 · Học bắt chước: robot học từ người lái (BC và DAgger) 🏎️

[![Mở trên Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanhpro82/tu-hoc-robot-tu-hanh/blob/main/tap46/tap46_dagger.ipynb)

Series **Tự Học Robot Tự Hành** · Mùa 4 · ⏱ ~40 phút · Giáo trình: Principles of Robot Autonomy, [Chương 19](https://porabook.com/book/imitation-learning/)

Robot xem chuyên gia lái rồi bắt chước; DAgger cho nó tự lái và được sửa. Trong notebook này bạn:

1. Đánh giá mạng đã học của video (BC 2 vòng mẫu vs DAgger).
2. **Tự viết** chuyên gia pure pursuit, behavior cloning và một vòng DAgger (3 bài), có ô chấm tự động (~30 giây).
3. Vẽ so sánh BC và DAgger.

Chạy lần lượt từng ô từ trên xuống (Shift + Enter).

## 0. Chuẩn bị (chạy 1 lần)

In [ ]:
import os, sys
if "google.colab" in sys.modules:  # trên Colab: tải code về
    if not os.path.exists("/content/tu-hoc-robot-tu-hanh"):
        !git clone -q https://github.com/thanhpro82/tu-hoc-robot-tu-hanh.git /content/tu-hoc-robot-tu-hanh
    os.chdir("/content/tu-hoc-robot-tu-hanh/tap46")
sys.path[:0] = ["bai_tap", "sim", ".."]

import json
import numpy as np
import duongdua as goc             # đường đua của video
from kiem_tra import nap
from ve_do_thi import ve           # vẽ hình
from IPython.display import Image, display
from kiem_tra import cham          # bộ chấm
print("Sẵn sàng!")

## 1. Chạy lại số liệu trong video

In [ ]:
k = json.load(open("sim/ket_qua.txt", encoding="utf8"))
print("số liệu video (500 đường, TB 4 seed): chuyên gia", k["chuyen_gia"], "% · BC 2 vòng", k["bc_ty"][0], "% · DAgger", k["dag_ty"][-1], "%")
for ten in ["dd_bc2_1.pt", "dd_dagger_1.pt"]:
    print(ten, goc.danh_gia(goc.lai_pi(nap(ten)), n=100))

## 2. Bài tập

Pure pursuit: $\omega = 2V\sin\alpha / L_d$. BC: cực tiểu $\frac1n\sum(\pi_\theta(x_i) - u_i)^2$. DAgger: người học tự lái → chuyên gia gán nhãn → gộp dữ liệu → học lại.

| Bài | Hàm | Việc |
|---|---|---|
| **1** | `omega_pure_pursuit(x, y, th, gx, gy, V, LD, W_MAX)` | lệnh quay của chuyên gia |
| **2** | `hoc_bc(X, U, seed, n_buoc, lr)` | học bắt chước bằng MSE |
| **3** | `vong_dagger(pi, env, X, U, hoc)` | một vòng DAgger |

Thay từng dòng `raise NotImplementedError(...)` bằng code của bạn, chạy ô đó, rồi chạy ô chấm ngay bên dưới (✅ đạt · ❌ sai, kèm gợi ý · ⏳ chưa làm).

In [ ]:
import math

import numpy as np
import torch

from duongdua import NguoiHoc, chay, lai_pi  # noqa: F401  (đường đua + mạng người học, trong ../sim)


def omega_pure_pursuit(x, y, th, gx, gy, V, LD, W_MAX):
    """Bài 1 (chuyên gia pure pursuit): robot ở (x, y), hướng th, nhắm điểm (gx, gy) trên đường tâm phía trước cách ~LD.

    α = góc từ hướng mũi xe tới điểm nhắm = atan2(gy − y, gx − x) − th, bọc về [−π, π)
    ω = 2·V·sin(α) / LD, kẹp trong [−W_MAX, W_MAX]. Trả về ω."""
    raise NotImplementedError("TODO Bài 1: pure pursuit")


def hoc_bc(X, U, seed=0, n_buoc=3000, lr=3e-3):
    """Bài 2 (behavior cloning, Thuật toán 1): học mạng bắt chước chuyên gia bằng hồi quy.

    torch.manual_seed(seed); pi = NguoiHoc(); opt = Adam(pi.parameters(), lr); g = torch.Generator().manual_seed(seed)
    Lặp n_buoc lần: j = torch.randint(len(X), (256,), generator=g)   (chọn ngẫu nhiên 256 mẫu)
                    loss = trung bình (pi(X[j]) − U[j])²;  opt.zero_grad(); loss.backward(); opt.step()
    (Đổi X, U sang tensor bằng torch.as_tensor trước vòng lặp.) Trả về pi."""
    raise NotImplementedError("TODO Bài 2: behavior cloning")


def vong_dagger(pi, env, X, U, hoc):
    """Bài 3 (DAgger, Thuật toán 3): một vòng.

    1. Cho NGƯỜI HỌC tự lái và chuyên gia gán nhãn:  r = chay(env, lai_pi(pi), nhan=True)   → r["X"], r["U"]
    2. Gộp dữ liệu: X = nối X với r["X"], U = nối U với r["U"]   (np.concatenate)
    3. Học lại từ đầu trên dữ liệu đã gộp: pi_moi = hoc(X, U)
    Trả về (pi_moi, X, U)."""
    raise NotImplementedError("TODO Bài 3: một vòng DAgger")


In [ ]:
cham(globals())

## 3. Xem kết quả

In [ ]:
ve(None, "ket_qua.png")
display(Image("ket_qua.png"))

## 4. Thử thách

BC càng nhiều vòng lái mẫu càng tốt? Đọc kết quả của 4 seed trong video.

In [ ]:
for s in range(4):
    d = json.load(open(f"sim/tn2_{s}.json", encoding="utf8"))
    bc = " ".join(f"{b['n_luot']}v:{b['ket']['xong'] * 100:3.0f}%" for b in d["bc"])
    print(f"seed {s}: BC {bc} | DAgger cuối {d['dagger'][-1]['ket']['xong'] * 100:.0f}%")

## Câu hỏi tự kiểm tra

Tự trả lời trước, rồi mở đáp án ở mục *Câu hỏi tự kiểm tra* trong [README của tập](https://github.com/thanhpro82/tu-hoc-robot-tu-hanh/tree/main/tap46).

Code cả series: [github.com/thanhpro82/tu-hoc-robot-tu-hanh](https://github.com/thanhpro82/tu-hoc-robot-tu-hanh)